# SIH NB13: frozen NB12b transfer evaluation on India

This notebook applies the frozen NB12b guarded CV plus tabular ensemble to a fixed India panel. It does not fit a model, tune a threshold, select a fusion weight, or alter the panel after seeing scores. EOG matches are positives. Unmatched sources are unlabelled, not verified negatives.

Use one T4 GPU and enable Internet. Attach the saved NB2 output containing `features_India_2022_2024.parquet`. The fixed panel contains 300 distinct 10 km blocks, with up to 96 distinct EOG sites. Image acquisition is capped at 100 new attempts per saved version. After each incomplete version, attach that version's output to the next run so the chips are reused.

India was previously evaluated with a superseded model. This is therefore a final-model transfer audit, not a first untouched project holdout and not a population-precision estimate.


In [ ]:
from pathlib import Path
import importlib.metadata
import json
import subprocess
import sys

INPUT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")
REPO = WORKING / "sih-appr"
ROOT = WORKING / "nb13_india_transfer"

required_versions = {
    "scikit-learn": "1.6.1",
    "lightgbm": "4.6.0",
    "joblib": "1.5.3",
}
install = [
    f"{package}=={required}"
    for package, required in required_versions.items()
    if importlib.metadata.version(package) != required
]
if install:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-deps", *install],
        check=True,
    )
import pandas as pd

if not (REPO / "kaggle/kg_13_india_guarded.py").is_file():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/ArnavLifelessCoder/sih-appr.git",
        str(REPO),
    ], check=True)
sys.path.insert(0, str(REPO / "kaggle"))

from kg_13_india_guarded import (
    bundle, export_image_features, prepare, run_batch, score_frozen_model
)

commit = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
print("Repository commit:", commit)


## Freeze the India panel before image acquisition

Country percentile ranks are computed on the full India common-window feature table before the enriched panel is sampled. Sampling is deterministic and uses one source per EOG site and one source per 10 km block.


In [ ]:
panel = prepare(
    INPUT,
    ROOT,
    n_sources=300,
    positive_site_quota=96,
    seed=13013,
)
display(panel.groupby("is_eog_flare").agg(
    sources=("source_id", "size"),
    blocks=("block_id", "nunique"),
    eog_sites=("eog_flare_id", "nunique"),
))
assert len(panel) == 300
assert panel.block_id.is_unique


## Reuse prior chips and download one bounded batch

The downloader first copies matching successful chips from any attached prior NB13 output. It then attempts at most 100 new sources or stops after 55 minutes. Failures remain visible and are not silently replaced.


In [ ]:
manifest = run_batch(
    ROOT,
    INPUT,
    max_new=100,
    max_minutes=55,
    retry_failed=False,
)
display(pd.crosstab(manifest.is_eog_flare, manifest.status))
display(manifest.loc[manifest.status.eq("failed"), [
    "source_id", "status", "error"
]] if "error" in manifest else manifest.head(0))


In [ ]:
image_features, image_quality = export_image_features(ROOT)
coverage = pd.read_csv(ROOT / "coverage_by_label.csv")
display(coverage)
state = json.loads((ROOT / "run_state.json").read_text(encoding="utf-8"))
print(json.dumps(state, indent=2))


## Score only when the fixed panel has no pending chips

On the final acquisition version, this cell downloads the hash-pinned Sentinel-2 encoder checkpoint, extracts the same CNN and morphology features used by NB12b, verifies all frozen artifact hashes, and scores the India panel. Earlier versions stop cleanly without exposing partial model metrics.


In [ ]:
pending = int(state.get("counts", {}).get("pending", 0))
if pending:
    print(f"{pending} chips remain. Save this version, attach its output to the next version, and rerun all cells.")
else:
    metrics, review_budgets, predictions = score_frozen_model(
        ROOT,
        REPO,
        checkpoint_path=None,
        bootstrap_repeats=2000,
    )
    display(metrics)
    display(review_budgets)
    display(json.loads((ROOT / "13_india_block_bootstrap.json").read_text(encoding="utf-8")))
    display(json.loads((ROOT / "13_india_decision.json").read_text(encoding="utf-8")))
    display(predictions.sort_values(
        "score_guarded_cv_tabular", ascending=False
    ).head(20))


In [ ]:
archive = bundle(ROOT)
print("Created:", archive)
if pending:
    print("Use Save Version and Save and Run All. Keep this saved output attached for the next acquisition version.")
else:
    print("India transfer evaluation is complete. Download the ZIP and share the full saved output.")
